# Full Dixon-Coles MLE

Fits team-specific attack (αᵢ) and defence (δⱼ) parameters via maximum likelihood,
rather than applying ρ on top of GLM-predicted λs.

This is the original Dixon-Coles (1997) formulation:
- Each team gets its own attack strength αᵢ and defence weakness δⱼ
- λ_home = exp(α_home - δ_away + γ)  where γ is home advantage
- λ_away = exp(α_away - δ_home)
- Joint PMF uses τ correction for low-score outcomes (same as notebook 12)
- Optional time decay: matches weighted by exp(-φ * days_since_match)

Key difference vs notebook 12:
- Notebook 12: GLM fits aggregate features → ρ correction on top
- Notebook 13: MLE fits per-team parameters directly → captures team heterogeneity the GLM cannot

Evaluation: same LOTO-CV over WC 2010/2014/2018/2022 (256 pooled matches).

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import mlflow
from scipy.stats import poisson
from scipy.optimize import minimize
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8
TIME_DECAY_PHI = 0.003  # ~0.3% weight decay per day; half-life ~230 days

In [2]:
def tau(h: int, a: int, lh: float, la: float, rho: float) -> float:
    """Dixon-Coles τ correction for the 2×2 low-score block."""
    if h == 0 and a == 0: return 1 - lh * la * rho
    if h == 1 and a == 0: return 1 + la * rho
    if h == 0 and a == 1: return 1 + lh * rho
    if h == 1 and a == 1: return 1 - rho
    return 1.0


def dc_joint_pmf(lh: float, la: float, rho: float, max_goals: int = MAX_GOALS) -> np.ndarray:
    """(max_goals+1)×(max_goals+1) joint PMF under Dixon-Coles."""
    goals = np.arange(max_goals + 1)
    ph = poisson.pmf(goals, lh)
    pa = poisson.pmf(goals, la)
    joint = np.outer(ph, pa)
    for hh in range(2):
        for aa in range(2):
            joint[hh, aa] *= tau(hh, aa, lh, la, rho)
    return joint


def best_score_dc(lh: float, la: float, rho: float) -> tuple:
    """(pred_h, pred_a) maximising expected Sporza pts under Dixon-Coles joint."""
    joint = dc_joint_pmf(lh, la, rho)
    best_pts, best_h, best_a = -1.0, 1, 0
    for ph in range(MAX_GOALS + 1):
        for pa in range(MAX_GOALS + 1):
            ep = 0.0
            for ah in range(MAX_GOALS + 1):
                for aa in range(MAX_GOALS + 1):
                    p = joint[ah, aa]
                    if p < 1e-9: continue
                    if ph == ah and pa == aa: ep += p * 10
                    elif (ph - pa) == (ah - aa): ep += p * 7
                    elif np.sign(ph - pa) == np.sign(ah - aa): ep += p * 5
                    else: ep += p * 1
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph, pa
    return best_h, best_a

In [3]:
class DixonColesModel:
    """
    Full Dixon-Coles MLE: fits per-team attack (alpha) and defence (delta)
    plus a home advantage (gamma) and ρ correction.

    λ_home = exp(alpha[home] - delta[away] + gamma)
    λ_away = exp(alpha[away] - delta[home])

    Identifiability: alpha[0] = 0 (fixed reference team).
    Free params: (n-1) alpha + n delta + gamma + rho = 2n+1.
    Likelihood is vectorised — no Python loop over matches.
    """

    def __init__(self, phi: float = TIME_DECAY_PHI):
        self.phi = phi
        self.teams_ = None
        self.team_idx_ = None
        self.alpha_ = None
        self.delta_ = None
        self.gamma_ = None
        self.rho_ = None

    def _weights(self, df: pd.DataFrame, ref_date) -> np.ndarray:
        days = (pd.Timestamp(ref_date) - pd.to_datetime(df['date'])).dt.days.values
        return np.exp(-self.phi * days)

    def fit(self, df: pd.DataFrame, ref_date=None):
        if ref_date is None:
            ref_date = df['date'].max()

        teams = sorted(set(df['home_team']) | set(df['away_team']))
        self.teams_ = teams
        self.team_idx_ = {t: i for i, t in enumerate(teams)}
        n = len(teams)

        hi = df['home_team'].map(self.team_idx_).values.astype(int)
        ai = df['away_team'].map(self.team_idx_).values.astype(int)
        hs = df['home_score'].values.astype(int)
        aw = df['away_score'].values.astype(int)
        w  = self._weights(df, ref_date)

        # Boolean masks for the 2x2 low-score block — precompute once
        m00 = (hs == 0) & (aw == 0)
        m10 = (hs == 1) & (aw == 0)
        m01 = (hs == 0) & (aw == 1)
        m11 = (hs == 1) & (aw == 1)
        low = m00 | m10 | m01 | m11

        def unpack_free(params_free):
            alpha = np.concatenate([[0.0], params_free[:n-1]])
            delta = params_free[n-1:2*n-1]
            gamma = params_free[2*n-1]
            rho   = float(np.clip(params_free[2*n], -0.99, 0.99))
            return alpha, delta, gamma, rho

        def neg_ll(params_free):
            alpha, delta, gamma, rho = unpack_free(params_free)
            lh = np.exp(alpha[hi] - delta[ai] + gamma)
            la = np.exp(alpha[ai] - delta[hi])

            # Poisson log-PMF: k*log(λ) - λ - log(k!)
            log_ph = hs * np.log(np.maximum(lh, 1e-15)) - lh - _log_factorial(hs)
            log_pa = aw * np.log(np.maximum(la, 1e-15)) - la - _log_factorial(aw)

            # τ correction (vectorised, only applied where needed)
            log_tau = np.zeros(len(hs))
            log_tau[m00] = np.log(np.maximum(1 - lh[m00] * la[m00] * rho, 1e-15))
            log_tau[m10] = np.log(np.maximum(1 + la[m10] * rho, 1e-15))
            log_tau[m01] = np.log(np.maximum(1 + lh[m01] * rho, 1e-15))
            log_tau[m11] = np.log(np.maximum(1 - rho, 1e-15))

            ll = w @ (log_tau + log_ph + log_pa)
            return -float(ll)

        # Precompute log-factorials up to max observed score
        _log_factorial_cache = np.zeros(max(hs.max(), aw.max()) + 1)
        for k in range(1, len(_log_factorial_cache)):
            _log_factorial_cache[k] = _log_factorial_cache[k-1] + np.log(k)

        def _log_factorial(arr):
            return _log_factorial_cache[arr]

        x0 = np.zeros(2*n + 1)
        x0[2*n - 1] = 0.3
        x0[2*n]     = -0.02
        bounds = ([(-3, 3)] * (n - 1) + [(-3, 3)] * n + [(-1, 3)] + [(-0.99, 0.99)])

        result = minimize(
            neg_ll, x0,
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 500, 'ftol': 1e-9}
        )

        self.alpha_, self.delta_, self.gamma_, self.rho_ = unpack_free(result.x)
        return self

    def predict_lambda(self, home_team: str, away_team: str, neutral: bool = False):
        hi = self.team_idx_.get(home_team)
        ai = self.team_idx_.get(away_team)
        if hi is None or ai is None:
            raise ValueError(f'Unknown team: {home_team!r} or {away_team!r}')
        gamma = 0.0 if neutral else self.gamma_
        lh = float(np.exp(self.alpha_[hi] - self.delta_[ai] + gamma))
        la = float(np.exp(self.alpha_[ai] - self.delta_[hi]))
        return lh, la

    def predict_score(self, home_team: str, away_team: str, neutral: bool = False):
        lh, la = self.predict_lambda(home_team, away_team, neutral)
        return best_score_dc(lh, la, self.rho_)

In [4]:
def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}

In [5]:
manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline (measured): {autofill_mean:.3f} pts/match')

Autofill baseline (measured): 4.137 pts/match


In [6]:
all_pts = []
fold_results = []
fold_models = {}

for year in WC_YEARS:
    print(f'\n--- WC {year} fold ---')
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    # Reference date: day before the eval tournament starts
    ref_date = evalf['date'].min() - pd.Timedelta(days=1)
    print(f'  Train: {len(train)} matches up to {train["date"].max().date()}')
    print(f'  Eval:  {len(evalf)} matches starting {evalf["date"].min().date()}')

    model = DixonColesModel(phi=TIME_DECAY_PHI)
    model.fit(train, ref_date=ref_date)
    fold_models[year] = model
    print(f'  γ (home adv) = {model.gamma_:.4f}  ρ = {model.rho_:.4f}')

    # Predict on eval fold
    preds = []
    missing_teams = []
    for _, row in evalf.iterrows():
        ht, at = row['home_team'], row['away_team']
        neutral = bool(row.get('is_neutral', 1))
        if ht not in model.team_idx_ or at not in model.team_idx_:
            missing_teams.append((ht, at))
            preds.append((1, 0))  # fallback: predict 1-0
        else:
            preds.append(model.predict_score(ht, at, neutral=neutral))

    if missing_teams:
        print(f'  WARNING: {len(missing_teams)} fixtures had unknown teams → fallback 1-0')
        for ht, at in missing_teams[:5]:
            print(f'    {ht} vs {at}')

    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])

    bd = score_breakdown(pred_home, pred_away,
                         evalf['home_score'].reset_index(drop=True),
                         evalf['away_score'].reset_index(drop=True))
    pts = sporza_points_series(pred_home, pred_away,
                               evalf['home_score'].reset_index(drop=True),
                               evalf['away_score'].reset_index(drop=True))
    all_pts.extend(pts.tolist())
    fold_results.append({
        'year': year,
        'gamma': model.gamma_,
        'rho': model.rho_,
        'mean_pts': bd['mean_pts'],
        'pct_exact': bd['pct_exact'],
        'pct_correct_result': bd['pct_correct_result'],
        'n_teams': len(model.teams_),
    })
    print(f'  → mean_pts={bd["mean_pts"]:.3f}  exact={bd["pct_exact"]:.1%}  '
          f'correct={bd["pct_correct_result"]:.1%}  teams_fit={len(model.teams_)}')

pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)
print(f'\nPooled LOTO-CV ({len(pooled_pts)} matches): {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]  '
      f'CI width={ci["ci_hi"]-ci["ci_lo"]:.3f}')


--- WC 2010 fold ---
  Train: 219 matches up to 2010-05-31
  Eval:  64 matches starting 2010-06-11


  γ (home adv) = 0.3618  ρ = -0.0921
    Brazil vs North Korea
    Brazil vs Ivory Coast
    Portugal vs Brazil
    Brazil vs Chile
    Netherlands vs Brazil
  → mean_pts=3.750  exact=9.4%  correct=48.4%  teams_fit=132

--- WC 2014 fold ---
  Train: 3900 matches up to 2014-05-31
  Eval:  64 matches starting 2014-06-12


  γ (home adv) = 0.2770  ρ = -0.0828
  → mean_pts=4.312  exact=12.5%  correct=56.2%  teams_fit=221

--- WC 2018 fold ---
  Train: 7272 matches up to 2018-05-31
  Eval:  64 matches starting 2018-06-14


  γ (home adv) = 0.3148  ρ = -0.0862
  → mean_pts=4.062  exact=9.4%  correct=54.7%  teams_fit=224

--- WC 2022 fold ---
  Train: 11106 matches up to 2022-10-30
  Eval:  64 matches starting 2022-11-20


  γ (home adv) = 0.2825  ρ = 0.0202
  → mean_pts=2.922  exact=4.7%  correct=37.5%  teams_fit=224

Pooled LOTO-CV (256 matches): 3.762 pts/match  95% CI [3.383, 4.141]  CI width=0.758


In [7]:
poisson_glm_mean = 4.336
dc_rho_only_mean = 4.328

print('Comparison vs all baselines:')
print(f'  Autofill (ELO-heuristic):        {autofill_mean:.3f} pts/match')
print(f'  Poisson GLM (notebook 11):       {poisson_glm_mean:.3f} pts/match  [3.930, 4.742]')
print(f'  Dixon-Coles ρ-only (nb 12):      {dc_rho_only_mean:.3f} pts/match  [3.926, 4.734]')
print(f'  Dixon-Coles full MLE (nb 13):    {ci["mean"]:.3f} pts/match  [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')
print(f'  Δ vs autofill:                   {ci["mean"] - autofill_mean:+.3f}')
print(f'  Δ vs Poisson GLM:                {ci["mean"] - poisson_glm_mean:+.3f}')
print(f'  Δ vs DC ρ-only:                  {ci["mean"] - dc_rho_only_mean:+.3f}')

beats_autofill = ci['ci_lo'] > autofill_mean
print(f'  CI lower > autofill:             {beats_autofill} → {"statistically beats" if beats_autofill else "not conclusive"}')

print('\nPer-fold results:')
for r in fold_results:
    print(f'  WC {r["year"]}: γ={r["gamma"]:+.3f}  ρ={r["rho"]:+.4f}  '
          f'mean_pts={r["mean_pts"]:.3f}  exact={r["pct_exact"]:.1%}  '
          f'correct={r["pct_correct_result"]:.1%}  teams={r["n_teams"]}')

Comparison vs all baselines:
  Autofill (ELO-heuristic):        4.137 pts/match
  Poisson GLM (notebook 11):       4.336 pts/match  [3.930, 4.742]
  Dixon-Coles ρ-only (nb 12):      4.328 pts/match  [3.926, 4.734]
  Dixon-Coles full MLE (nb 13):    3.762 pts/match  [3.383, 4.141]
  Δ vs autofill:                   -0.375
  Δ vs Poisson GLM:                -0.574
  Δ vs DC ρ-only:                  -0.566
  CI lower > autofill:             False → not conclusive

Per-fold results:
  WC 2010: γ=+0.362  ρ=-0.0921  mean_pts=3.750  exact=9.4%  correct=48.4%  teams=132
  WC 2014: γ=+0.277  ρ=-0.0828  mean_pts=4.312  exact=12.5%  correct=56.2%  teams=221
  WC 2018: γ=+0.315  ρ=-0.0862  mean_pts=4.062  exact=9.4%  correct=54.7%  teams=224
  WC 2022: γ=+0.282  ρ=+0.0202  mean_pts=2.922  exact=4.7%  correct=37.5%  teams=224


In [8]:
# Top attack / defence teams from the most recent fold (2022)
model_2022 = fold_models[2022]
alpha = pd.Series(model_2022.alpha_, index=model_2022.teams_).sort_values(ascending=False)
delta = pd.Series(model_2022.delta_, index=model_2022.teams_).sort_values(ascending=True)

print('Top 10 attack (alpha, higher = better scorer):')
print(alpha.head(10).to_string())
print('\nTop 10 defence (delta, lower = harder to score against):')
print(delta.head(10).to_string())

Top 10 attack (alpha, higher = better scorer):
Brazil         1.730001
Argentina      1.355868
Uruguay        1.182281
Colombia       1.176559
Portugal       1.040852
Belgium        1.032152
Netherlands    1.031034
Japan          1.017086
Serbia         1.004612
Germany        0.996609

Top 10 defence (delta, lower = harder to score against):
Turks and Caicos Islands           -1.883126
British Virgin Islands             -1.570147
Brunei                             -1.439344
Anguilla                           -1.430879
Tuvalu                             -1.428846
Laos                               -1.394220
Guam                               -1.273665
Cayman Islands                     -1.196388
Saint Vincent and the Grenadines   -1.157194
Cambodia                           -1.151157


In [9]:
mlflow.set_tracking_uri('sqlite:////Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/mlflow.db')
mlflow.set_experiment('wk2026-tabular-2026-06-13')

with mlflow.start_run(run_name='dixon_coles_full_mle_loto'):
    mlflow.set_tags({
        'modality': 'tabular',
        'approach': 'dixon_coles_full_mle',
        'eval_strategy': 'loto_cv',
        'dataset_version': 'split_v2'
    })
    mlflow.log_params({
        'time_decay_phi': TIME_DECAY_PHI,
        'optimizer': 'L-BFGS-B',
        'score_strategy': 'max_expected_sporza_pts',
        'max_goals_grid': MAX_GOALS,
        'identifiability': 'alpha_0_fixed_0',
    })
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    mlflow.log_metric('autofill_baseline', autofill_mean)
    mlflow.log_metric('delta_vs_autofill', ci['mean'] - autofill_mean)
    mlflow.log_metric('delta_vs_poisson_glm', ci['mean'] - poisson_glm_mean)
    mlflow.log_metric('delta_vs_dc_rho_only', ci['mean'] - dc_rho_only_mean)
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_exact', r['pct_exact'])
        mlflow.log_metric(f'fold_{r["year"]}_gamma', r['gamma'])
        mlflow.log_metric(f'fold_{r["year"]}_rho', r['rho'])
    run_id = mlflow.active_run().info.run_id

print(f'run_id: {run_id}')

run_id: f556a9b547634c7e973e4d8c6049f18d
